# Mathematical Foundations for ML — Module 1 Foundations

## Learning Objectives
- Master the linear algebra fundamentals underlying all ML algorithms
- Understand probability distributions and their role in model-based reasoning
- Implement gradient descent and optimization from first principles
- Connect mathematical concepts to practical ML workflows

## Why This Matters
Skillsoft Track 1 includes 3h22m of Linear Algebra + Probability courses covering vector spaces, matrix operations, eigenvalues/eigenvectors, PCA, probability distributions, CLT, parameter estimation, and binomial/Gaussian distributions. These foundations are prerequisite for everything from linear regression to deep learning.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from functools import reduce
from operator import mul
import warnings
warnings.filterwarnings('ignore')
sns.set_style("whitegrid")
np.random.seed(42)
%matplotlib inline


## 1. Linear Algebra Fundamentals

### 1.1 Vectors and Vector Spaces

A **vector** is an element of a vector space, represented as an ordered list of numbers. In ML, vectors represent data points, features, or model parameters.


In [ ]:
# Basic vector operations
v = np.array([3.0, 1.0, 4.0, 1.0, 5.0])
w = np.array([2.0, 7.0, 1.0, 8.0, 9.0])

print("Vector v:", v)
print("Vector w:", w)
print("v + w (element-wise addition):", v + w)
print("2 * v (scalar multiplication):", 2 * v)
print("Dot product v·w:", np.dot(v, w))
print("L2 norm ||v||:", np.linalg.norm(v))


### 1.2 Matrices and Matrix Operations

A **matrix** is a 2D array of numbers. Matrix multiplication is the most important operation in ML.


In [ ]:
# Matrix creation
A = np.array([[1.0, 2.0], [3.0, 4.0], [5.0, 6.0]]),  # 3x2 matrix
B = np.array([[7.0, 8.0, 9.0], [10.0, 11.0, 12.0]]),  # 2x3 matrix

print("Matrix A shape:", A.shape)
print("Matrix B shape:", B.shape)
print("A @ B (matrix multiplication):\n", A @ B)
print("Element-wise multiply:\n", A * B)

print("A transpose:\n", A.T)

I = np.eye(3)
print("3x3 Identity:\n", I)

A_square = np.array([[1.0, 2.0], [3.0, 4.0]])
try:
    A_inv = np.linalg.inv(A_square)
    print("A inverse:\n", A_inv)
    print("A @ A^{-1} =:\n", A_square @ A_inv)
except np.linalg.LinAlgError:
    print("Matrix is singular, cannot invert")


### 1.3 Eigenvalues and Eigenvectors

An **eigenvector** of a matrix A is a non-zero vector v such that Av = λv, where λ is the eigenvalue.


In [ ]:
# Compute eigenvalues and eigenvectors
matrix = np.array([[2.0, 1.0], [1.0, 2.0]])
eigenvalues, eigenvectors = np.linalg.eig(matrix)

print("Matrix:\n", matrix)
print("Eigenvalues:", eigenvalues)
print("Eigenvectors:\n", eigenvectors)

# Verify: A @ v = λ * v
for i, lam in enumerate(eigenvalues):
    v = eigenvectors[:, i]
    print(f"λ={lam:.4f}, A@v = {A @ v}, λ*v = {lam * v}, match={np.allclose(A @ v, lam * v)}")


### 1.4 Singular Value Decomposition (SVD)

SVD factorizes a matrix A into U @ Σ @ V^T. It's the mathematical backbone of PCA.


In [ ]:
# SVD demonstration
X = np.array([
    [1, 2],
    [3, 4],
    [5, 6],
], dtype=float)

U, S, VT = np.linalg.svd(X, full_matrices=False)
print("U (left singular vectors):\n", U)
print("Singular values:", S)
print("V^T (right singular vectors):\n", VT)
print("Reconstruction A ≈ U @ diag(S) @ V^T:\n", U @ np.diag(S) @ VT)

# Rank approximation (keep top k singular values)
k = 1
X_approx = U[:, :k] @ np.diag(S[:k]) @ VT[:k, :]
print(f"\nRank-{k} approximation error (frobenius): {np.linalg.norm(X - X_approx, 'fro'):.4f}")
print("Original X:\n", X)
print(f"Rank-{k} approximation:\n", X_approx)


### 1.5 PCA from SVD

PCA via SVD of the data matrix (centered).


In [ ]:
def pca_svd(X, k):
    """Return top-k principal components and transformed data."""
    X_centered = X - X.mean(axis=0)
    U, S, VT = np.linalg.svd(X_centered, full_matrices=False)
    components = VT[:k].T  # shape (n_features, k)
    transformed = X_centered @ components  # shape (n_samples, k)
    return components, transformed, S[:k]

n_samples, n_features = 200, 5
X = np.random.randn(n_samples, n_features)
components, transformed, singular_values = pca_svd(X, k=2)

print(f"Original shape: {X.shape}")
print(f"Transformed shape: {transformed.shape}")
print(f"Top-{k} singular values: {singular_values}")
print(f"Explained variance ratio: {(singular_values**2 / np.sum(singular_values**2)):.4f}")


## 2. Probability Distributions and Statistical Foundations

### 2.1 Common Distributions


In [ ]:
import scipy.stats as stats
x = np.linspace(stats.norm.ppf(0.001), stats.norm.ppf(0.999), 1000)

fig, axes = plt.subplots(2, 3, figsize=(15, 8))

# Normal distribution
axes[0,0].plot(x, stats.norm.pdf(x, 0, 1))
axes[0,0].set_title('Normal(0,1) PDF')
axes[0,0].set_ylabel('Density')

# Standard normal CDF
axes[0,1].plot(x, stats.norm.cdf(x, 0, 1))
axes[0,1].set_title('Normal(0,1) CDF')

# Binomial distribution
x_bin = np.arange(0, 30)
axes[1,0].plot(x_bin, stats.binom.pmf(x_bin, n=25, p=0.5))
axes[1,0].set_title('Binomial(n=25, p=0.5) PMF')
axes[1,0].set_xlabel('Number of successes')

# Poisson distribution
axes[1,1].plot(x_bin, stats.poisson.pmf(x_bin, mu=3))
axes[1,1].set_title('Poisson(μ=3) PMF')
axes[1,1].set_xlabel('k events')

# Uniform distribution
axes[1,2].plot([0, 1], [1, 1], linewidth=2)
axes[1,2].set_title('Uniform(0,1) PDF')
axes[1,2].set_ylim(0, 1.5)

plt.tight_layout()
plt.show()


### 2.2 Law of Large Numbers and Central Limit Theorem


In [ ]:
# Demonstrate CLT with various original distributions
np.random.seed(123)
n_trials = 10000
n_samples_list = [1, 5, 30, 100]

means = {}
for n in n_samples_list:
    samples = np.random.uniform(0, 1, (n_trials, n))
    sample_means = samples.mean(axis=1)
    means[n] = sample_means
    print(f"n={n}: mean of sample means = {sample_means.mean():.4f} (expected: 0.5), "
          f"std of sample means = {sample_means.std():.4f} (expected: ~{np.sqrt(1/12)/np.sqrt(n):.4f})")

plt.figure(figsize=(10, 6))
for n, sample_means in means.items():
    plt.hist(sample_means, bins=50, density=True, alpha=0.6, label=f'n={n}', histtype='step')
    x = np.linspace(0.3, 0.7, 200)
    plt.plot(x, stats.norm.pdf(x, 0.5, np.sqrt(1/12)/np.sqrt(n)), 'k--', 
             label=f'N(0.5, {np.sqrt(1/12)/np.sqrt(n):.4f})')

plt.title('Central Limit Theorem Demonstration')
plt.xlabel('Sample mean')
plt.ylabel('Density')
plt.legend()
plt.show()


### 2.3 Bayes' Theorem


In [ ]:
# Bayes' theorem example: spam detection
p_spam = 0.1
p_not_spam = 0.9
p_free_given_spam = 0.8
p_free_given_ham = 0.1

p_free = p_free_given_spam * p_spam + p_free_given_ham * p_not_spam
p_spam_given_free = (p_free_given_spam * p_spam) / p_free

print(f"P(spam) = {p_spam}")
print(f"P(free|spam) = {p_free_given_spam}")
print(f"P(free|not spam) = {p_free_given_ham}")
print(f"P(free) = {p_free:.4f}")
print(f"P(spam|free) = {p_spam_given_free:.4f}")
print(f"Given 'free' appears, probability email is spam: {p_spam_given_free:.1%}")


### 2.4 Parameter Estimation: MLE and MAP


In [ ]:
def mle_bernoulli(heads, total):
    """MLE: θ = heads / total"""
    return heads / total

def map_bernoulli(heads, total, alpha_prior=1, beta_prior=1):
    """MAP with Beta(alpha, beta) prior: θ = (heads + alpha) / (total + alpha + beta)"""
    return (heads + alpha_prior) / (total + alpha_prior + beta_prior)

heads, total = 7, 10
mle_theta = mle_bernoulli(heads, total)
map_theta = map_bernoulli(heads, total, alpha_prior=1, beta_prior=1)  # Uniform prior

print(f"Data: {heads} heads in {total} flips")
print(f"MLE estimate: θ = {mle_theta:.4f} ({heads/total*100:.1f}%)")
print(f"MAP estimate (uniform prior): θ = {map_theta:.4f}")
print(f"MAP estimate (weak prior α=2, β=2): θ = {map_bernoulli(heads, total, alpha_prior=2, beta_prior=2):.4f}")
print(f"MAP estimate (strong prior α=5, β=5): θ = {map_bernoulli(heads, total, alpha_prior=5, beta_prior=5):.4f}")


## 3. Gradient Descent and Optimization


In [ ]:
def f(x):
    return x**2

def df(x):
    return 2*x

# Gradient descent from x = 10
x = 10.0
learning_rate = 0.1
n_iterations = 20

history = [x]
for i in range(n_iterations):
    grad = df(x)
    x = x - learning_rate * grad
    history.append(x)
    if i < 5 or i >= n_iterations - 3:
        print(f"Iter {i}: x = {x:.4f}, gradient = {grad:.4f}")

print(f"\nFinal x = {x:.4f} (should approach 0)")
print(f"Convergence {'✓' if abs(x) < 0.01 else '✗'}")

xs = np.linspace(-12, 12, 200)
plt.plot(xs, xs**2, label='f(x) = x²', color='gray')
plt.plot(history, [f(x) for x in history], 'ro-', label='GD path')
plt.xlabel('x')
plt.ylabel('f(x)')
plt.title('Gradient Descent on f(x) = x²')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


### 3.3 Learning Rate Sensitivity


In [ ]:
def run_gd(lr, n_iter=50):
    x = 10.0
    history = [x]
    for i in range(n_iter):
        grad = df(x)
        x = x - lr * grad
        history.append(x)
    return history

lrs = [0.001, 0.01, 0.05, 0.1, 0.2, 0.5, 1.0]
results = {lr: run_gd(lr) for lr in lrs}

plt.figure(figsize=(10, 6))
for lr, hist in results.items():
    plt.plot(hist, label=f'lr={lr}')

plt.axhline(0, color='k', linestyle='--', alpha=0.3)
plt.xlabel('Iteration')
plt.ylabel('x value')
plt.title('Learning Rate Sensitivity in Gradient Descent')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


## 4. Exercises


### Exercise 1: Vector Operations and Similarity

Given two document vectors, compute cosine similarity.


In [ ]:
def cosine_similarity(v, w):
    """cosine_similarity(v, w) = (v·w) / (||v|| ||w||)"""
    return np.dot(v, w) / (np.linalg.norm(v) * np.linalg.norm(w))

doc1 = np.array([1, 0, 1, 1, 0, 1])
doc2 = np.array([0, 1, 1, 0, 1, 1])
doc3 = np.array([1, 1, 0, 1, 0, 1])

print(f"cosine(doc1, doc2) = {cosine_similarity(doc1, doc2):.4f} (expected low, dissimilar)")
print(f"cosine(doc1, doc3) = {cosine_similarity(doc1, doc3):.4f} (expected high, similar)")


### Exercise 2: PCA Dimensionality Reduction

Apply PCA to reduce from 5 dimensions to 2, and verify explained variance.


### Exercise 3: Bayesian Coin Flip


In [ ]:
np.random.seed(123)
heads, total = 15, 20
mle_theta = heads / total
map_uniform = (heads + 1) / (total + 2)  # Uniform prior α=1, β=1
map_beta5 = (heads + 5) / (total + 10)  # Strong prior α=5, β=5

print(f"Data: {heads} heads in {total} flips")
print(f"MLE estimate: θ = {mle_theta:.4f} ({heads/total*100:.1f}%)")
print(f"MAP estimate (uniform prior): θ = {map_uniform:.4f}")
print(f"MAP estimate (strong prior α=5, β=5): θ = {map_beta5:.4f}")
print(f"MAP estimates are 'closer to 0.5' when prior is stronger")


### Exercise 4: Gradient Descent Convergence


lrs = [0.001, 0.01, 0.05, 0.1, 0.2]
results = {lr: run_gd(lr, n_iter=50) for lr in lrs}

max_lr_converges = max(lr for lr, hist in results.items() if abs(hist[-1]) < 0.01)
print(f"Maximum learning rate that converges: {max_lr_converges}")


## 5. Summary: Math → ML Pipeline

| Mathematical Concept | ML Application | Module Notebook |
|---------------------|----------------|-----------------|
| Vector dot product | Similarity scoring | 01_data_preprocessing.ipynb |
| Matrix multiplication | Forward propagation | 02_experiment_tracking.ipynb |
| Eigenvalues/eigenvectors | PCA, dimensionality reduction | **04_mathematical_foundations.ipynb** |
| SVD | Recommender systems, matrix factorization | **04_mathematical_foundations.ipynb** |
| PDF/CDF/CLT | Uncertainty quantification | **04_mathematical_foundations.ipynb** |
| Bayes' theorem | Bayesian inference | **04_mathematical_foundations.ipynb** |
| Gradient descent | Linear regression, model training | 03_model_evaluation.ipynb |
| MLE/MAP | Parameter estimation | **04_mathematical_foundations.ipynb** |

Key Takeaway: The mathematics in this notebook are not abstract — they are the engine under every ML algorithm you'll build. Master these foundations, and the transition from 'implementing code' to 'designing models' becomes natural.


**Next Steps**:
- Review the exercise solutions above
- Connect each mathematical concept to the notebooks in this module
- Practice implementing algorithms from first principles (don't just use sklearn!)
- Move to Module 2 to learn how to operationalize these models in production